In [12]:
import torch
import torch.nn as nn
import torchvision
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
from torchvision import datasets
import pandas as pd

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.RandomHorizontalFlip(p = 0.5), #randomly mirroring images
    transforms.RandomRotation(degrees = 15), # rotate up to 15 degrees 
    transforms.ColorJitter(
        brightness=0.3, 
        contrast=0.3,
        saturation=0.3
    ),
    transforms.ToTensor(), 
    transforms.Normalize([0.5]*3,[0.5]*3)
])

#validation/test transform has no augmentation
val_test_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [3]:
import os

base_dir = "data"
splits = ["seg_train", "seg_test"]

class_counts_all = {}

for split in splits:
    data_dir = os.path.join(base_dir, split)
    class_counts = {}

    for class_name in os.listdir(data_dir):
        class_path = os.path.join(data_dir, class_name)

        if os.path.isdir(class_path):
            jpg_files = [
                file for file in os.listdir(class_path)
                if file.lower().endswith(".jpg")
            ]

            class_counts[class_name] = len(jpg_files)

    class_counts_all[split] = class_counts

print("class counts:\n")
print(class_counts_all)

class counts:

{'seg_train': {'buildings': 2191, 'forest': 2271, 'glacier': 2404, 'mountain': 2512, 'sea': 2274, 'street': 2382}, 'seg_test': {'buildings': 437, 'forest': 474, 'glacier': 553, 'mountain': 525, 'sea': 510, 'street': 501}}


In [4]:
train_dir = "data/seg_train/"
test_dir = "data/seg_test/"

In [5]:
full_train = datasets.ImageFolder(root=train_dir,transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_dir,transform=val_test_transform)

In [6]:
# 85% train, 15% val
train_size = int(0.85* len(full_train))
val_size = len(full_train) - train_size

train_dataset, val_dataset = random_split(full_train, [train_size, val_size])

val_dataset.dataset = datasets.ImageFolder(root=train_dir, transform=val_test_transform)


In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [8]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(18*18*128, 256), 
            nn.ReLU(), 
            nn.Dropout(0.5), 
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 6)
        )


    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

In [9]:
class EarlyStopping:
    def __init__(self, patience=7):
        self.patience = patience
        self.counter = 0
        self.best_loss = float("inf")
        self.stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min', 
    factor=0.5,
    patience=3,
    verbose=True
)

Using: cuda


C:\Users\Hp\anaconda3\envs\ml\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [11]:
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_loss  = float("inf")
early_stopping = EarlyStopping(patience=7)
epochs = 50

for epoch in range(epochs):
    # ── Training ──────────────────────────────────────────
    model.train()
    epoch_training_loss      = 0.0
    correct_train, total_train = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss   = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()
        _, predicted = torch.max(output, 1)
        correct_train += (predicted == labels).sum().item()
        total_train   += labels.size(0)

    epoch_train_loss = epoch_training_loss / len(train_loader)
    epoch_train_acc  = 100 * correct_train / total_train
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc)

    # ── Validation ────────────────────────────────────────
    model.eval()
    running_val_loss         = 0.0
    correct_val, total_val   = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            loss   = criterion(output, labels)

            running_val_loss += loss.item()
            _, predicted = torch.max(output, 1)
            correct_val  += (predicted == labels).sum().item()
            total_val    += labels.size(0)

    epoch_val_loss = running_val_loss / len(val_loader)
    epoch_val_acc  = 100 * correct_val / total_val
    val_losses.append(epoch_val_loss)
    val_accs.append(epoch_val_acc)

    print(f"Epoch {epoch+1}/{epochs} → "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")

    # ── LR Scheduler step ─────────────────────────────────
    scheduler.step(epoch_val_loss)    # checks if val loss improved, reduces LR if not

    # ── Save Best ─────────────────────────────────────────
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt")
        print(f"  ✓ Best model saved at epoch {epoch+1}")

    # ── Early Stopping check ──────────────────────────────
    early_stopping(epoch_val_loss)
    if early_stopping.stop:
        print(f"\n  ✗ Early stopping triggered at epoch {epoch+1}")
        print(f"  Best val loss was {best_val_loss:.4f}")
        break

Epoch 1/50 → Train Loss: 1.1827 | Train Acc: 52.54% | Val Loss: 0.8896 | Val Acc: 64.58%
  ✓ Best model saved at epoch 1
Epoch 2/50 → Train Loss: 0.8816 | Train Acc: 66.74% | Val Loss: 0.7799 | Val Acc: 69.75%
  ✓ Best model saved at epoch 2
Epoch 3/50 → Train Loss: 0.7579 | Train Acc: 72.70% | Val Loss: 0.6488 | Val Acc: 76.21%
  ✓ Best model saved at epoch 3
Epoch 4/50 → Train Loss: 0.6836 | Train Acc: 75.88% | Val Loss: 0.6580 | Val Acc: 74.74%
Epoch 5/50 → Train Loss: 0.6301 | Train Acc: 78.40% | Val Loss: 0.5588 | Val Acc: 81.10%
  ✓ Best model saved at epoch 5
Epoch 6/50 → Train Loss: 0.5860 | Train Acc: 80.22% | Val Loss: 0.5171 | Val Acc: 82.15%
  ✓ Best model saved at epoch 6
Epoch 7/50 → Train Loss: 0.5413 | Train Acc: 81.77% | Val Loss: 0.5381 | Val Acc: 81.29%
Epoch 8/50 → Train Loss: 0.5202 | Train Acc: 82.29% | Val Loss: 0.5295 | Val Acc: 81.20%
Epoch 9/50 → Train Loss: 0.4827 | Train Acc: 82.75% | Val Loss: 0.4858 | Val Acc: 83.29%
  ✓ Best model saved at epoch 9
Epoch 1

In [13]:

metrics_df = pd.DataFrame({
    "epoch": range(1, len(train_losses) + 1),
    "train_loss": train_losses,
    "val_loss": val_losses,
    "train_acc": train_accs,
    "val_acc": val_accs
})

metrics_df.to_csv("training_metrics.csv", index=False)

print("Training metrics saved to training_metrics.csv")

Training metrics saved to training_metrics.csv
